In [1]:
from utils import dataloader

import yaml

import numpy as np
import pandas as pd

from tqdm import tqdm

import nibabel as nib

/home/petron/AIMS/Thesis/TRACE_Algonauts/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


[fetch_atlas_schaefer_2018] Dataset found in /home/petron/nilearn_data/schaefer_2018


In [2]:
with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

# Data Loading Test

#### Get Stimuli Info +  Load Transcript

In [6]:
# @title Get Stimuli Info per subject

subject = 1

fmri_stimuli_info = dataloader.list_fmri_sessions(
    dir=FMRI_DIR,
    subject=subject,
    split='train'
)

In [ ]:
fmri_stimuli_info

In [9]:
fmri_stimuli_info[0][13:]

's01e02a'

In [10]:
# @title Load stimuli transcript

transcript_df = dataloader.load_transcript(
    dir = TRANSCRIPT_DIR, 
    stimuli_name=fmri_stimuli_info[0],
    split='train',
    ignore_nans=True
    )

transcript_df.shape

(366, 4)

In [11]:
transcript_df

,text_per_tr,words_per_tr,onsets_per_tr,durations_per_tr
0,You.,['You.'],[0.33],[0.55]
1,What you guys don't,"['What', 'you', 'guys', ""don't""]","[1.89, 2.228, 2.388, 2.612]","[0.316, 0.138, 0.186, 0.214]"
2,"understand is, for us,","['understand', 'is,', 'for', 'us,']","[2.858, 3.22, 3.556, 3.812]","[0.292, 0.282, 0.218, 0.586]"
3,kissing is as important,"['kissing', 'is', 'as', 'important']","[4.564, 5.098, 5.268, 5.428]","[0.502, 0.148, 0.138, 0.33]"
4,as any part of it.,"['as', 'any', 'part', 'of', 'it.']","[5.844, 6.068, 6.292, 6.468, 6.628]","[0.202, 0.186, 0.154, 0.138, 0.522]"
...,...,...,...,...
473,She's,"[""She's""]",[705.37],[0.68]
474,pregnant with my,"['pregnant', 'with', 'my']","[706.13, 706.706, 707.036]","[0.512, 0.276, 0.218]"
475,child.,['child.'],[707.292],[0.588]
476,And she and Susan are going,"['And', 'she', 'and', 'Susan', 'are', 'going']","[708.97, 709.436, 709.628, 709.788, 710.178, 7...","[0.412, 0.17, 0.138, 0.358, 0.196, 0.154]"


#### Load Subject fMRI data per stimuli & subject atlas

In [13]:
# @title Load subject fMRI resposnes data for a specific stimuli

subject = 5

recording_session = '001'
season = '01'
episode = '02'
episode_split = 'b'

fmri_key = f'ses-{recording_session}_task-s{season}e{episode}{episode_split}'
print(f'stimuli+session: {fmri_key}')

file_name = "sub-01_task-friends_space-MNI152NLin2009cAsym_atlas-Schaefer18_parcel-1000Par7Net_desc-s123456_bold.h5"
fmri_data = dataloader.load_fmri_responses(
    dir=FMRI_DIR,
    subject=subject,
    stimuli_name=fmri_key,
    split='train'
)
fmri_data.shape

stimuli+session: ses-001_task-s01e02b


(482, 1000)

In [23]:
# @title Fetah each subject's atlas info

import yaml

with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

with open('configs/configs.yaml', 'r') as f:
    configs = yaml.safe_load(f)

HRF_DELAY = configs['hrf_delay']
TR = configs['tr']
CONTEXT_TRS = configs['context_trs']

SUBJECTS = configs['subjects']
print(f'subjects: {SUBJECTS}')

subject1_atlas_coord, parcel_ids, parcel_desc = dataloader.load_atlas_for_subject(subject=1, dir=FMRI_DIR)
subject2_atlas_coord, parcel_ids, parcel_desc = dataloader.load_atlas_for_subject(subject=2, dir=FMRI_DIR)
subject3_atlas_coord, parcel_ids, parcel_desc = dataloader.load_atlas_for_subject(subject=3, dir=FMRI_DIR)
subject5_atlas_coord, parcel_ids, parcel_desc = dataloader.load_atlas_for_subject(subject=5, dir=FMRI_DIR)

(subject1_atlas_coord == subject5_atlas_coord).all()

subjects: [1, 2, 3, 5]


np.True_

## Subject data-loading test

In [4]:
import yaml

with open('configs/dirs.yaml', 'r') as f:
    dir_configs = yaml.safe_load(f)

TRANSCRIPT_DIR = dir_configs['TRANSCRIPT_DIR']
FMRI_DIR = dir_configs['FMRI_DIR']

with open('configs/configs.yaml', 'r') as f:
    configs = yaml.safe_load(f)

HRF_DELAY = configs['hrf_delay']
TR = configs['tr']
CONTEXT_TRS = configs['context_trs']

SUBJECTS = configs['subjects']
SUBJECTS

[1, 2, 3, 5]

#### Stimuli-Episodes fMRI Response

In [27]:
subject = SUBJECTS[0]
SPLIT = 'train'
subject_dataset = dataloader.load_episode_fmri(subject, split=SPLIT, fmri_dir=FMRI_DIR)

Loading fMRI timeseries for s-01


100%|██████████| 292/292 [00:02<00:00, 110.10it/s]


Loading parcel coordinates for s-01


In [ ]:
for stimuli_name in subject_dataset['scenes_response']:
    print(stimuli_name, subject_dataset['scenes_response'][stimuli_name].shape)

#### Word Epoched (Stimuli-Episodes) fMRI Response

In [5]:
subject = SUBJECTS[0]
SPLIT = 'train'
subject_dataset = dataloader.load_episode_word_epochs(
    subject, split=SPLIT, 
    fmri_dir=FMRI_DIR, transcript_dir=TRANSCRIPT_DIR,
    hrf_delay=HRF_DELAY, tr=TR, context_trs=CONTEXT_TRS)

Loading word epochs for s-01


100%|██████████| 292/292 [00:14<00:00, 19.83it/s]


Loading parcel coordinates for s-01


In [6]:
subject_dataset.keys()

dict_keys(['scenes_response', 'parcel_coords', 'parcel_ids', 'parcel_desc'])

In [ ]:
subject_dataset['parcel_desc']

In [ ]:
for stimuli_name in subject_dataset['scenes_response'].keys():
    print(
        stimuli_name,
        subject_dataset['scenes_response'][stimuli_name]['trials'].shape, 
        (subject_dataset['scenes_response'][stimuli_name]['start'].shape, subject_dataset['scenes_response'][stimuli_name]['end'].shape)
        )